# Final Project

This final project can be collaborative. The maximum members of a group is 3. You can also work by yourself. Please respect the academic integrity. **Remember: if you get caught on cheating, you get F.**

## A Introduction to the competition

<img src="news-sexisme-EN.jpg" alt="drawing" width="380"/>

Sexism is a growing problem online. It can inflict harm on women who are targeted, make online spaces inaccessible and unwelcoming, and perpetuate social asymmetries and injustices. Automated tools are now widely deployed to find, and assess sexist content at scale but most only give classifications for generic, high-level categories, with no further explanation. Flagging what is sexist content and also explaining why it is sexist improves interpretability, trust and understanding of the decisions that automated tools use, empowering both users and moderators.

This project is based on SemEval 2023 - Task 10 - Explainable Detection of Online Sexism (EDOS). [Here](https://codalab.lisn.upsaclay.fr/competitions/7124#learn_the_details-overview) you can find a detailed introduction to this task.

You only need to complete **TASK A - Binary Sexism Detection: a two-class (or binary) classification where systems have to predict whether a post is sexist or not sexist**. To cut down training time, we only use a subset of the original dataset (5k out of 20k). The dataset can be found in the same folder. 

Different from our previous homework, this competition gives you great flexibility (and very few hints). You can freely determine every component of your workflow, including but not limited to:
-  **Preprocessing the input text**: You may decide how to clean or transform the text. For example, removing emojis or URLs, lowercasing, removing stopwords, applying stemming or lemmatization, correcting spelling, or performing tokenization and sentence segmentation.
-  **Feature extraction and encoding**: You can choose any method to convert text into numerical representations, such as TF-IDF, Bag-of-Words, N-grams, Word2Vec, GloVe, FastText, contextual embeddings (e.g., BERT, RoBERTa, or other transformer-based models), Part-of-Speech (POS) tagging, dependency-based features, sentiment or emotion features, readability metrics, or even embeddings or features generated by large language models (LLMs).
-  **Data augmentation and enrichment**: You may expand or balance your dataset by incorporating other related corpora or using techniques like synonym replacement, random deletion/insertion, or LLM-assisted augmentation (e.g., generating paraphrased or synthetic examples to improve model robustness).
-  **Model selection**: You are free to experiment with different models — from traditional machine learning algorithms (e.g., Logistic Regression, SVM, Random Forest, XGBoost) to deep learning architectures (e.g., CNNs, RNNs, Transformers), or even hybrid/ensemble approaches that combine multiple models or leverage LLM-generated predictions or reasoning.

## Requirements
-  **Input**: the text for each instance.
-  **Output**: the binary label for each instance.
-  **Feature engineering**: use at least 2 different methods to extract features and encode text into numerical values. You may explore both traditional and AI-assisted techniques. Data augmentation is optional.
-  **Model selection**: implement with at least 3 different models and compare their performance.
-  **Evaluation**: create a dataframe with rows indicating feature+model and columns indicating Precision (P), Recall (R) and F1-score (using weighted average). Your results should have at least 6 rows (2 feature engineering methods x 3 models). Report best performance with (1) your feature engineering method, and (2) the model you choose. Here is an example illustrating how the experimental results table should be presented.

| Feature + Model | Sexist (P) | Sexist (R) | Sexist (F1) | Non-Sexist (P) | Non-Sexist (R) | Non-Sexist (F1) | Weighted (P) | Weighted (R) | Weighted (F1) |
|-----------------|:----------:|:----------:|:------------:|:---------------:|:---------------:|:----------------:|:-------------:|:--------------:|:---------------:|
| TF-IDF + Logistic Regression | ... | ... | ... | ... | ... | ... | ... | ... | ... |

- **Format of the report**: add explainations for each step (you can add markdown cells). At the end of the report, write a summary for each sections: 
    - Data Preprocessing
    - Feature Engineering
    - Model Selection and Architecture
    - Training and Validation
    - Evaluation and Results
    - Use of Generative AI (if you use)

## Rules 
Violations will result in 0 points in the grade: 
-   `Rule 1 - No test set leakage`: You must not use any instance from the test set during training, feature engineering, or model selection.
-   `Rule 2 - Responsible AI use`: You may use generative AI, but you must clearly document how it was used. If you have used genAI, include a section titled “Use of Generative AI” describing:
    -   What parts of the project you used AI for
    -   What was implemented manually vs. with AI assistance

## Grading

The performance should be only evaluated on the test set (a total of 1086 instances). Please split original dataset into train set and test set. The test set should NEVER be used in the training process. The evaluation metric is a combination of precision, recall, and f1-score (use `classification_report` in sklearn). 

The total points are 10.0. Each team will compete with other teams in the class on their best performance. Points will be deducted if not following the requirements above. 

If ALL the requirements are met:
- Top 25\% teams: 10.0 points.
- Top 25\% - 50\% teams: 8.5 points.
- Top 50\% - 75\% teams: 7.0 points.
- Top 75\% - 100\% teams: 6.0 points.

If your best performance reaches **0.82** or above (weighted F1-score) and follows all the requirements and rules, you will also get full points (10.0 points). 

## Submission
Similar as homework, submit both a PDF and .ipynb version of the report including: 
- code and experimental results with details explained
- combined results table, report and best performance
- a summary at the end of the report (please follow the format above)

Missing any part of the above requirements will result in point deductions.

The due date is **Dec 11, Thursday by 11:59pm**.

In [ ]:
# Preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import pandas as pd

nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stop_words(inp):
    words = nltk.word_tokenize(str(inp))
    no_stop_words = []
    for word in words:
        if word not in stop_words:
            no_stop_words.append(word)
    return ' '.join(no_stop_words)

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = nltk.word_tokenize(str(text))
    lemmas = [lemmatizer.lemmatize(w) for w in words]
    return ' '.join(lemmas)

data = pd.read_csv("edos_labelled_data.csv")

# make the columns mirror the original data given
# keep only the columns you want and rename label_sexist -> label
data = data[['rewire_id', 'text', 'label_sexist', 'split']]
data = data.rename(columns={'label_sexist': 'label'})


# apply the same preprocessing
# Lowercase
data['text'] = data['text'].str.lower()
# Remove URLs
data['text'] = data['text'].str.replace(r'http\S+|www\S+', '', regex=True)
# Remove HTML tags
data['text'] = data['text'].str.replace(r'<.*?>', '', regex=True)
# Remove remaining non-alphanumeric characters (emojis, punctuation)
data['text'] = data['text'].str.replace(r'[^a-zA-Z0-9 ]', '', regex=True)
# Remove stop words
data['text'] = data['text'].apply(remove_stop_words)
# Lemmatize
data['text'] = data['text'].apply(lemmatize_text)

data.to_csv("preprocessed_edos_labelled_data.csv", index=False)

In [15]:
import pandas as pd
resultsDF = pd.DataFrame(columns=["Feature + Model", "Sexist (P)", "Sexist (R)", "Sexist (F1)", "Non-Sexist (P)", "Non-Sexist (R)", "Non-Sexist (F1)", "Weighted (P)", "Weighted (R)", "Weighted (F1)"], index=['tl', 'ts', 'tr', 'bl', 'bs', 'br'])
pd.options.display.float_format = '{:.2f}'.format

In [16]:
# TF-IDF + Logistic Regression
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TunedThresholdClassifierCV
from sklearn.metrics import classification_report

print("Loading data...")
df = pd.read_csv("preprocessed_edos_labelled_data.csv")  # change path if needed
# Make sure text is a string and has no NaNs
df['text'] = df['text'].fillna("")      # replace NaN with empty string
df['text'] = df['text'].astype(str)     # force everything to be string


encoder = LabelEncoder()
y = encoder.fit_transform(df['label']) # 0 for not sexist, 1 for sexist

# mask data
train_mask = df['split'] == 'train'
test_mask = df['split'] == 'test'

# get tweet lists
X_train_tweets = df.loc[train_mask, 'text']
X_test_tweets = df.loc[test_mask, 'text']

# get labels
y_train = y[train_mask]
y_test = y[test_mask]

# tf-idf vectorization
print("Encoding text with TF-IDF...")
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2), use_idf=True, sublinear_tf=True, stop_words='english', min_df=2)

X_train = vectorizer.fit_transform(X_train_tweets)
X_test = vectorizer.transform(X_test_tweets)

# LogisticRegression model
classifier = LogisticRegression(max_iter=1000, C=1, solver='liblinear', class_weight={0: 1, 1: 4})
tuned_model = TunedThresholdClassifierCV(estimator=classifier, scoring='f1_weighted')

tuned_model.fit(X_train, y_train)
print(f"Best Threshold Found: {tuned_model.best_threshold_}")
y_pred = tuned_model.predict(X_test)
cr = classification_report(y_test, y_pred, target_names=encoder.classes_, output_dict=True)
resultsDF.loc['tl'] = pd.Series({"Feature + Model":"TF-IDF+LogisticRegression", "Sexist (P)":cr['sexist']['precision'], "Sexist (R)":cr['sexist']['recall'], "Sexist (F1)":cr['sexist']['f1-score'], "Non-Sexist (P)":cr['not sexist']['precision'], "Non-Sexist (R)":cr['not sexist']['recall'], "Non-Sexist (F1)":cr['not sexist']['f1-score'], "Weighted (P)":cr['weighted avg']['precision'], "Weighted (R)":cr['weighted avg']['recall'], "Weighted (F1)":cr['weighted avg']['f1-score']})

Loading data...
Encoding text with TF-IDF...
Best Threshold Found: 0.6106504472532455


In [ ]:
# TF-IDF + SVM
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.model_selection import TunedThresholdClassifierCV
from sklearn.metrics import classification_report

print("Loading data...")
df = pd.read_csv("preprocessed_edos_labelled_data.csv")  # change path if needed

encoder = LabelEncoder()
y = encoder.fit_transform(df['label']) # 0 for not sexist, 1 for sexist
# Make sure text is a string and has no NaNs
df['text'] = df['text'].fillna("")      # replace NaN with empty string
df['text'] = df['text'].astype(str)     # force everything to be string


# mask data
train_mask = df['split'] == 'train'
test_mask = df['split'] == 'test'

# get tweet lists
X_train_tweets = df.loc[train_mask, 'text']
X_test_tweets = df.loc[test_mask, 'text']

# get labels
y_train = y[train_mask]
y_test = y[test_mask]

# tf-idf vectorization
print("Encoding text with TF-IDF...")
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), use_idf=True, sublinear_tf=True)

X_train = vectorizer.fit_transform(X_train_tweets)
X_test = vectorizer.transform(X_test_tweets)

# Support Vector Machine model
classifier = LinearSVC(max_iter=1000, C=.1, loss='hinge')
tuned_model = TunedThresholdClassifierCV(estimator=classifier, scoring='f1_weighted')

tuned_model.fit(X_train, y_train)
print(f"Best Threshold Found: {tuned_model.best_threshold_}")
y_pred = tuned_model.predict(X_test)
cr = classification_report(y_test, y_pred, target_names=encoder.classes_, output_dict=True)
resultsDF.loc['ts'] = pd.Series({"Feature + Model":"TF-IDF+SVM", "Sexist (P)":cr['sexist']['precision'], "Sexist (R)":cr['sexist']['recall'], "Sexist (F1)":cr['sexist']['f1-score'], "Non-Sexist (P)":cr['not sexist']['precision'], "Non-Sexist (R)":cr['not sexist']['recall'], "Non-Sexist (F1)":cr['not sexist']['f1-score'], "Weighted (P)":cr['weighted avg']['precision'], "Weighted (R)":cr['weighted avg']['recall'], "Weighted (F1)":cr['weighted avg']['f1-score']})

Loading data...
Encoding text with TF-IDF...
Best Threshold Found: -0.8651913339382827
              Feature + Model Sexist (P) Sexist (R) Sexist (F1)  \
tl  TF-IDF+LogisticRegression       0.64       0.60        0.62   
ts                 TF-IDF+SVM       0.80       0.45        0.57   
tr                        NaN        NaN        NaN         NaN   
bl                        NaN        NaN        NaN         NaN   
bs                        NaN        NaN        NaN         NaN   
br                        NaN        NaN        NaN         NaN   

   Non-Sexist (P) Non-Sexist (R) Non-Sexist (F1) Weighted (P) Weighted (R)  \
tl           0.87           0.89            0.88         0.82         0.82   
ts           0.84           0.96            0.90         0.83         0.84   
tr            NaN            NaN             NaN          NaN          NaN   
bl            NaN            NaN             NaN          NaN          NaN   
bs            NaN            NaN             NaN     

In [ ]:
# TF-IDF + Random Forest
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

print("Loading data...")
df = pd.read_csv("preprocessed_edos_labelled_data.csv")  # change path if needed

encoder = LabelEncoder()
y = encoder.fit_transform(df['label']) # 0 for not sexist, 1 for sexist
# Make sure text is a string and has no NaNs
df['text'] = df['text'].fillna("")      # replace NaN with empty string
df['text'] = df['text'].astype(str)     # force everything to be string


# mask data
train_mask = df['split'] == 'train'
test_mask = df['split'] == 'test'

# get tweet lists
X_train_tweets = df.loc[train_mask, 'text']
X_test_tweets = df.loc[test_mask, 'text']

# get labels
y_train = y[train_mask]
y_test = y[test_mask]

# tf-idf vectorization
print("Encoding text with TF-IDF...")
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), use_idf=True, sublinear_tf=True)

X_train = vectorizer.fit_transform(X_train_tweets)
X_test = vectorizer.transform(X_test_tweets)

# Support Vector Machine model
classifier = RandomForestClassifier(random_state=0)

classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
cr = classification_report(y_test, y_pred, target_names=encoder.classes_, output_dict=True)
resultsDF.loc['tr'] = pd.Series({"Feature + Model":"TF-IDF+RandomForest", "Sexist (P)":cr['sexist']['precision'], "Sexist (R)":cr['sexist']['recall'], "Sexist (F1)":cr['sexist']['f1-score'], "Non-Sexist (P)":cr['not sexist']['precision'], "Non-Sexist (R)":cr['not sexist']['recall'], "Non-Sexist (F1)":cr['not sexist']['f1-score'], "Weighted (P)":cr['weighted avg']['precision'], "Weighted (R)":cr['weighted avg']['recall'], "Weighted (F1)":cr['weighted avg']['f1-score']})

In [ ]:
# Bert + Logistic Regression
import pandas as pd
from sentence_transformers import SentenceTransformer, models
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

print("Loading data...")
df = pd.read_csv("preprocessed_edos_labelled_data.csv")  # Change path if needed

encoder = LabelEncoder()
y = encoder.fit_transform(df['label']) # 0 for not sexist, 1 for sexist

# Load BERT embedder
print("\nLoading bert model...")
bert = models.Transformer('bert-base-uncased', max_seq_length=128)   # 128 tokens otherwise model might not be as accurate, doc said model trained on 128
pooling_model = models.Pooling(bert.get_word_embedding_dimension())  # take the average of the word vectors
embedder = SentenceTransformer(modules=[bert, pooling_model])        # combine bert and the pooler into one embedder

print("Generating embeddings...")
X = embedder.encode(df['text'].tolist())

train_tweets = df['split'] == 'train'
test_tweets = df['split'] == 'test' 

# Apply masks to extract train and test tweets
X_train = X[train_tweets]
X_test = X[test_tweets]

encoder = LabelEncoder()
y = encoder.fit_transform(df['label'])

y_train = y[train_tweets]
y_test = y[test_tweets]

classifier = LogisticRegression(max_iter=2000) # iterate much longer because bert vectors way larger

print("\nTraining classifier...")
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)
cr = classification_report(y_test, y_pred, target_names=encoder.classes_, output_dict=True)
resultsDF.loc['bl'] = pd.Series({"Feature + Model":"TF-IDF+SVM", "Sexist (P)":cr['sexist']['precision'], "Sexist (R)":cr['sexist']['recall'], "Sexist (F1)":cr['sexist']['f1-score'], "Non-Sexist (P)":cr['not sexist']['precision'], "Non-Sexist (R)":cr['not sexist']['recall'], "Non-Sexist (F1)":cr['not sexist']['f1-score'], "Weighted (P)":cr['weighted avg']['precision'], "Weighted (R)":cr['weighted avg']['recall'], "Weighted (F1)":cr['weighted avg']['f1-score']})

In [ ]:
# Bert + SVM
import pandas as pd
from sklearn import svm
from sentence_transformers import SentenceTransformer, models
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

print("Loading data...")
df = pd.read_csv("preprocessed_edos_labelled_data.csv")  # Change path if needed

encoder = LabelEncoder()
y = encoder.fit_transform(df['label']) # 0 for not sexist, 1 for sexist

# Load BERT embedder
print("\nLoading bert model...")
bert = models.Transformer('bert-base-uncased', max_seq_length=128)   # 128 tokens otherwise model might not be as accurate, doc said model trained on 128
pooling_model = models.Pooling(bert.get_word_embedding_dimension())  # take the average of the word vectors
embedder = SentenceTransformer(modules=[bert, pooling_model])        # combine bert and the pooler into one embedder

print("Generating embeddings...")
X = embedder.encode(df['text'].tolist())

train_tweets = df['split'] == 'train'
test_tweets = df['split'] == 'test' 

# Apply masks to extract train and test tweets
X_train = X[train_tweets]
X_test = X[test_tweets]

encoder = LabelEncoder()
y = encoder.fit_transform(df['label'])

y_train = y[train_tweets]
y_test = y[test_tweets]

weights = {0:5288/(5288-1565), 1:5288/1565}

classifier = svm.SVC(kernel='linear', class_weight=weights)

print("\nTraining classifier...")
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)
cr = classification_report(y_test, y_pred, target_names=encoder.classes_, output_dict=True)
resultsDF.loc['br'] = pd.Series({"Feature + Model":"TF-IDF+SVM", "Sexist (P)":cr['sexist']['precision'], "Sexist (R)":cr['sexist']['recall'], "Sexist (F1)":cr['sexist']['f1-score'], "Non-Sexist (P)":cr['not sexist']['precision'], "Non-Sexist (R)":cr['not sexist']['recall'], "Non-Sexist (F1)":cr['not sexist']['f1-score'], "Weighted (P)":cr['weighted avg']['precision'], "Weighted (R)":cr['weighted avg']['recall'], "Weighted (F1)":cr['weighted avg']['f1-score']})

In [ ]:
# Bert + Random Forest
import pandas as pd
from sklearn import ensemble
from sentence_transformers import SentenceTransformer, models
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

print("Loading data...")
df = pd.read_csv("preprocessed_edos_labelled_data.csv")  # Change path if needed

encoder = LabelEncoder()
y = encoder.fit_transform(df['label']) # 0 for not sexist, 1 for sexist

# Load BERT embedder
print("\nLoading bert model...")
bert = models.Transformer('bert-base-uncased', max_seq_length=128)   # 128 tokens otherwise model might not be as accurate, doc said model trained on 128
pooling_model = models.Pooling(bert.get_word_embedding_dimension())  # take the average of the word vectors
embedder = SentenceTransformer(modules=[bert, pooling_model])        # combine bert and the pooler into one embedder

print("Generating embeddings...")
X = embedder.encode(df['text'].tolist())

train_tweets = df['split'] == 'train'
test_tweets = df['split'] == 'test' 

# Apply masks to extract train and test tweets
X_train = X[train_tweets]
X_test = X[test_tweets]

encoder = LabelEncoder()
y = encoder.fit_transform(df['label'])

y_train = y[train_tweets]
y_test = y[test_tweets]

weights = {0:5288/(5288-1565), 1:5288/1565}

classifier = ensemble.RandomForestClassifier(random_state=0)

print("\nTraining classifier...")
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)
cr = classification_report(y_test, y_pred, target_names=encoder.classes_, output_dict=True)
resultsDF.loc['bs'] = pd.Series({"Feature + Model":"TF-IDF+SVM", "Sexist (P)":cr['sexist']['precision'], "Sexist (R)":cr['sexist']['recall'], "Sexist (F1)":cr['sexist']['f1-score'], "Non-Sexist (P)":cr['not sexist']['precision'], "Non-Sexist (R)":cr['not sexist']['recall'], "Non-Sexist (F1)":cr['not sexist']['f1-score'], "Weighted (P)":cr['weighted avg']['precision'], "Weighted (R)":cr['weighted avg']['recall'], "Weighted (F1)":cr['weighted avg']['f1-score']})


In [ ]:
print(resultsDF)

## Experimental Results

(A table detailed model performance on the test set with at least 6 rows. Report the best performance.)


## Project Summary
### 1. Data Preprocessing


### 2. Feature Engineering
 

### 3. Model Selection and Architecture


### 4. Training and Validation


### 5. Evaluation and Results


### 6. Use of Generative AI (if you use)